# ML Phil — URL Phishing Detection: end-to-end pipeline

One notebook: **imports → data → EDA → preprocessing → features → training → evaluation → stability tests → export**.

**Kaggle settings:** Internet **ON**, CPU is enough (≈30–45 min). Optional: add a dataset with `urls.json` / `hf_phishing_urls.json`, `majestic_million.csv`, `top-1m.csv(.zip)` to skip downloads.

**Design rules** (from the audit of earlier attempts):
1. The model reads the **URL string only** — no submitted URL is ever opened, fetched or resolved.
2. Dataset artefacts are removed before modelling: scheme, leading `www.`, trailing `/` (they correlated with the label).
3. Splits are **grouped by registered domain** (or full hostname for shared platforms like `pages.dev`) — no domain is in two splits.
4. External cohorts are **domain-disjoint** from all training data. Thresholds are chosen on *dev* data only; *test* is read once.
5. Output is a **risk band** (HIGH / MEDIUM / LOW). LOW means "no URL-pattern warning", never "safe".

**For the ML Phil website:** after **Save & Run All (Commit)**, download `model_bundle.zip` (now includes `final_report.json`) and `ext_test_urls.csv.gz`. The bundle also contains `predictor.py`, `urlcore.py` and `features.py`, so the site can load it unchanged.

In [ ]:
# Same versions as the website (backend/requirements.txt), so the exported model scores identically there.
!pip install -q lightgbm==4.7.0 tldextract==5.4.0 rapidfuzz==3.14.6 statsmodels

## 1. Imports & configuration

In [ ]:
import os, re, io, gc, sys, json, math, time, glob, random, hashlib, zipfile, ipaddress, warnings
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt
import requests, tldextract, lightgbm as lgb, sklearn
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.preprocessing import normalize
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 90)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# '/kaggle' alone is not enough: on Windows it means C:\kaggle, which can exist on a laptop.
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/input').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path(os.environ.get('MLPHIL_WORK', 'mlphil_work')).resolve()
DATA, OUT, FIG = WORK / 'data', WORK / 'model_bundle', WORK / 'figures'
for d in (DATA, OUT, FIG): d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK); sys.path.insert(0, str(WORK))

CFG = dict(
    PDB_TRAIN_ROWS=200_000,        # modern phishing added to training (domains disjoint from external cohorts)
    PDB_TEST_ROWS_PER_HALF=12_000, # external phishing rows per dev/test half
    PYPI_PACKAGES=15_000,          # top PyPI packages scanned for modern benign links
    PYPI_WEIGHT=8.0,               # up-weight modern benign links (small source)
    MAJESTIC_CAP=180_000,
    TRANCO_TOP=15_000,             # popular homepages used as an external benign-proxy cohort
    TEXT_FOLDS=4, SGD_ALPHA=2e-6,
    GBM_LR=0.05, GBM_LEAVES=63, GBM_MIN_CHILD=200, GBM_MAX_ROUNDS=3000,
    ALERT_MAX_FPR=0.03, REVIEW_MAX_FPR=0.10,
    STABILITY_SEEDS=5, CV_FOLDS=5, BOOTSTRAP=1000,
    HF_FRACTION=1.0,               # <1.0 only for low-RAM machines
)
CFG.update(json.loads(os.environ.get('MLPHIL_CFG', '{}')))      # optional overrides
LOCAL = json.loads(os.environ.get('MLPHIL_LOCAL', '{}'))        # optional local file paths

# Acceptance criteria are fixed HERE, before any training (evaluated once on the test cohorts).
CRITERIA = {
    'ext phishing recall (PhishingDatabase) >= 0.80': ('ext_test', 'PhishingDatabase_ACTIVE', '>=', 0.80),
    'ext phishing recall (OpenPhish) >= 0.75':        ('ext_test', 'OpenPhish_feed', '>=', 0.75),
    'ext benign false alert (Tranco homepages) <= 0.05': ('ext_test', 'Tranco_roots', '<=', 0.05),
    'ext benign false alert (curated links) <= 0.05':    ('ext_test', 'Curated_link_lists', '<=', 0.05),
    'internal test benign false alert <= 0.05':          ('test', 'ALL_benign', '<=', 0.05),
    'internal test phishing recall >= 0.85':             ('test', 'ALL_phishing', '>=', 0.85),
}
T0 = time.time()
print('work dir:', WORK, '| kaggle:', ON_KAGGLE)
print({m.__name__: m.__version__ for m in (np, pd, sklearn, lgb, tldextract)})

## 2. Data acquisition

| Source | Role | Label quality |
|---|---|---|
| HuggingFace `ealvaradob/phishing-dataset` → `urls.json` | main training set | dataset labels |
| Majestic Million | benign homepages (TLD-balanced) for training | popularity proxy |
| Phishing.Database `ACTIVE` (GitHub) | modern phishing: part training, part external test | feed-listed |
| OpenPhish public feed (GitHub) | external phishing test | feed-listed |
| Tranco top list | external benign homepages | popularity proxy |
| Human-curated link lists (GitHub awesome / free-books lists) | external benign deep links | human-curated proxy |
| PyPI README / project links | modern benign deep links for training | maintainer-published proxy |

Only **list files** are downloaded. No URL inside them is visited.

In [ ]:
UA = {'User-Agent': 'mlphil-research-notebook'}
PROV = {}

def find_input(*names):
    for n in names:
        hits = glob.glob(f'/kaggle/input/**/{n}', recursive=True)
        if hits: return Path(hits[0])

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def get(name, url, dest, local_names=()):
    """local override -> Kaggle input -> cached download -> download"""
    path = Path(LOCAL[name]) if name in LOCAL else find_input(*local_names)
    if path is None:
        path = DATA / dest
        if not path.exists() or path.stat().st_size == 0:
            r = requests.get(url, headers=UA, timeout=900); r.raise_for_status(); path.write_bytes(r.content)
    PROV[name] = dict(path=str(path), url=url, bytes=path.stat().st_size, sha256=sha256(path),
                      recorded_utc=datetime.now(timezone.utc).isoformat(timespec='seconds'))
    return path

P_HF  = get('hf_urls', 'https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/urls.json', 'urls.json', ('urls.json', 'hf_phishing_urls.json'))
P_MAJ = get('majestic', 'https://downloads.majestic.com/majestic_million.csv', 'majestic_million.csv', ('majestic_million.csv',))
P_TRA = get('tranco', 'https://tranco-list.eu/top-1m.csv.zip', 'top-1m.csv.zip', ('top-1m.csv.zip', 'top-1m.csv'))
P_PDB = get('phishing_database', 'https://raw.githubusercontent.com/Phishing-Database/Phishing.Database/master/phishing-links-ACTIVE.txt', 'pdb_active.txt', ('phishing-links-ACTIVE.txt',))
P_OPF = get('openphish', 'https://raw.githubusercontent.com/openphish/public_feed/refs/heads/main/feed.txt', 'openphish_feed.txt')

CURATED = """EbookFoundation/free-programming-books/main/books/free-programming-books-langs.md
EbookFoundation/free-programming-books/main/books/free-programming-books-subjects.md
EbookFoundation/free-programming-books/main/books/free-programming-books-ru.md
EbookFoundation/free-programming-books/main/books/free-programming-books-ja.md
EbookFoundation/free-programming-books/main/books/free-programming-books-zh.md
EbookFoundation/free-programming-books/main/books/free-programming-books-de.md
EbookFoundation/free-programming-books/main/books/free-programming-books-fr.md
EbookFoundation/free-programming-books/main/books/free-programming-books-es.md
EbookFoundation/free-programming-books/main/books/free-programming-books-pt_BR.md
EbookFoundation/free-programming-books/main/books/free-programming-books-it.md
EbookFoundation/free-programming-books/main/books/free-programming-books-ko.md
EbookFoundation/free-programming-books/main/books/free-programming-books-pl.md
EbookFoundation/free-programming-books/main/books/free-programming-books-tr.md
EbookFoundation/free-programming-books/main/books/free-programming-books-id.md
EbookFoundation/free-programming-books/main/books/free-programming-books-hi.md
EbookFoundation/free-programming-books/main/courses/free-courses-en.md
EbookFoundation/free-programming-books/main/courses/free-courses-ru.md
EbookFoundation/free-programming-books/main/courses/free-courses-es.md
EbookFoundation/free-programming-books/main/courses/free-courses-pt_BR.md
EbookFoundation/free-programming-books/main/more/free-programming-cheatsheets.md
EbookFoundation/free-programming-books/main/more/free-programming-interactive-tutorials-en.md
public-apis/public-apis/master/README.md
sindresorhus/awesome/main/readme.md
vinta/awesome-python/master/README.md
avelino/awesome-go/main/README.md
awesome-selfhosted/awesome-selfhosted/master/README.md
jaywcjlove/awesome-mac/master/README-zh.md
enaqx/awesome-react/master/README.md
ripienaar/free-for-dev/master/README.md
trimstray/the-book-of-secret-knowledge/master/README.md
kahun/awesome-sysadmin/master/README.md
josephmisiti/awesome-machine-learning/master/README.md
awesomedata/awesome-public-datasets/master/README.rst""".split()
cur_dir = Path(LOCAL['curated_dir']) if 'curated_dir' in LOCAL else DATA / 'curated'
cur_dir.mkdir(exist_ok=True)
for p in CURATED:
    f = cur_dir / p.replace('/', '_')
    if not f.exists():
        try:
            r = requests.get('https://raw.githubusercontent.com/' + p, headers=UA, timeout=120); r.raise_for_status(); f.write_bytes(r.content)
        except Exception as e: print('curated skip', p, e)
PROV['curated'] = dict(files=len(list(cur_dir.glob('*'))), source='raw.githubusercontent.com (list above)')

# PyPI README links (modern benign deep URLs for training)
URL_RX = r'https?://[^\s)\]>"\'`<|]+'
P_PYPI = Path(LOCAL['pypi_links']) if 'pypi_links' in LOCAL else DATA / 'pypi_links.txt'
if not P_PYPI.exists():
    top = requests.get('https://raw.githubusercontent.com/hugovk/top-pypi-packages/main/top-pypi-packages.min.json', headers=UA, timeout=120).json()
    pkgs = [r['project'] for r in top['rows'][:CFG['PYPI_PACKAGES']]]
    def pypi_links(pkg):
        try:
            d = requests.get(f'https://pypi.org/pypi/{pkg}/json', headers=UA, timeout=30).json()['info']
        except Exception: return []
        u = list((d.get('project_urls') or {}).values()) + ([d['home_page']] if d.get('home_page') else [])
        return u + [m.rstrip('.,;:*_') for m in re.findall(URL_RX, (d.get('description') or '')[:200_000])]
    with ThreadPoolExecutor(32) as ex: links = [l for ls in ex.map(pypi_links, pkgs) for l in ls if l]
    P_PYPI.write_text('\n'.join(links))
PROV['pypi_links'] = dict(path=str(P_PYPI), sha256=sha256(P_PYPI))
json.dump(PROV, open(WORK / 'data_provenance.json', 'w'), indent=1)
pd.DataFrame(PROV).T[['bytes', 'sha256', 'recorded_utc']].dropna(how='all')

## 3. Shared URL canonicalisation & feature code
Written to `.py` files so training, evaluation and the deployed predictor use the **exact same code**.

In [ ]:
%%writefile mlphil_urlcore.py
"""ML Phil URL model — shared URL canonicalisation + feature code.

Used identically by training, evaluation and inference (single source of truth).
URL string analysis only: this module never resolves, fetches or opens a URL.
"""
from __future__ import annotations
import ipaddress, math, re
from collections import Counter
from functools import lru_cache
from urllib.parse import urlsplit, parse_qsl
import numpy as np
import tldextract

_EXT = tldextract.TLDExtract(suffix_list_urls=())   # bundled PSL snapshot, no network
_SCHEME = re.compile(r'^[a-z][a-z0-9+.-]*://', re.I)

# Shared-hosting / user-content platforms: a tenant subdomain is a separate site,
# so grouping + reputation must use the full hostname, not the registered domain.
PLATFORMS = frozenset('''
blogspot.com blogger.com wordpress.com wixsite.com wix.com weebly.com weeblysite.com
000webhostapp.com firebaseapp.com web.app github.io gitlab.io netlify.app vercel.app
pages.dev workers.dev r2.dev glitch.me herokuapp.com azurewebsites.net appspot.com
cloudfront.net amazonaws.com ngrok.io ngrok-free.app ngrok.app duckdns.org no-ip.org
ddns.net hopto.org zapto.org sites.google.com godaddysites.com square.site webflow.io
framer.app framer.website craftum.io tilda.ws mystrikingly.com strikingly.com jimdo.com
jimdofree.com yolasite.com site123.me ucoz.ru ucoz.net narod.ru tripod.com
angelfire.com webnode.page webnode.com ipfs.io dweb.link cloudflare-ipfs.com
myshopify.com repl.co replit.dev replit.app onrender.com fly.dev surge.sh
backblazeb2.com storage.googleapis.com windows.net sharepoint.com
1drv.ms my.canva.site canva.site notion.site typedream.app carrd.co
firebasestorage.googleapis.com page.link qrco.de qrfy.io linktr.ee
contaboserver.net blob.core.windows.net web.core.windows.net
'''.split())

SHORTENERS = frozenset('''bit.ly tinyurl.com t.co goo.gl ow.ly is.gd buff.ly cutt.ly rb.gy
rebrand.ly shorturl.at tiny.cc s.id v.gd bl.ink t.ly lnkd.in qr.ae'''.split())

# Frequently impersonated brands (lower-case tokens). A brand token in the
# hostname/path while the registered domain is NOT that brand is a classic lure.
BRANDS = frozenset('''
paypal apple icloud itunes microsoft office365 office outlook hotmail live onedrive
sharepoint netflix amazon aws prime chase wellsfargo bankofamerica citibank citi
hsbc barclays santander natwest lloyds halifax dhl fedex usps ups royalmail auspost
postnl swisspost laposte correos poste canadapost instagram facebook meta whatsapp
telegram messenger coinbase binance kraken metamask blockchain trustwallet ledger
steam steamcommunity epicgames roblox adobe docusign dropbox linkedin google gmail
yahoo aol att verizon tmobile xfinity comcast spectrum irs hmrc gov ebay walmart
costco bestbuy americanexpress amex mastercard visa discover capitalone usaa
navyfederal sbi hdfc icici axis paytm phonepe kotak airtel jio netbanking
alibaba aliexpress taobao rakuten mercadolibre mercadopago nubank bradesco itau
caixa santander bancodobrasil sparkasse volksbank postbank ing rabobank abnamro
kbc belfius bnpparibas societegenerale creditagricole orange free sfr bouygues
vodafone o2 telstra optus allegro olx booking airbnb uber zoom webex slack
discord twitter x tiktok snapchat spotify yandex mailru vk sberbank tinkoff
wetransfer godaddy namecheap cpanel webmail roundcube zimbra
'''.split())
_GENERIC = {'x', 'free', 'live', 'meta', 'gov', 'axis', 'discover', 'prime', 'office', 'orange', 'messenger', 'ledger', 'booking', 'ing', 'poste', 'citi'}
BRANDS = frozenset(BRANDS - _GENERIC)
_BRAND_SHORT_OK = {'o2', 'aol', 'ups', 'kbc', 'sbi', 'sfr', 'ebay', 'visa', 'jio', 'vk', 'aws', 'irs', 'usps', 'dhl', 'att', 'amex', 'uber', 'zoom', 'olx'}
_BRAND_MATCH = frozenset(b for b in BRANDS if len(b) >= 5)  # substring match only for longer tokens

LURE_WORDS = frozenset('''login logon signin sign-in verify verification secure account accounts update
password passwd credential wallet billing invoice payment confirm recover recovery reset
auth authenticate suspend suspended unlock support bonus gift giveaway crypto claim
reward refund prize unusual activity limited restore validate validation webscr
session token otp kyc banking customer service helpdesk alert notice security'''.split())

# Marketing / click-tracking parameters carry no information about the destination -> removed.
_TRACKING = re.compile(r'^(utm_[a-z_]+|fbclid|gclid|gbraid|wbraid|dclid|msclkid|yclid|mc_cid|mc_eid|_hsenc|_hsmi|igshid|mkt_tok|oly_anon_id|oly_enc_id|vero_id|_ga|_gl|ref_src)$')

FILE_EXT = re.compile(r'\.(php|html?|aspx?|jsp|cgi|exe|zip|apk|scr|js)$', re.I)


class BadURL(ValueError):
    pass


@lru_cache(maxsize=200_000)
def _ext(host: str):
    return _EXT(host)


def parse(raw: str) -> dict:
    """Canonicalise a URL string into scheme-free, www-free parts.

    Training datasets disagree on whether URLs carry a scheme, 'www.' or a
    trailing slash, and those artefacts correlate with the label (proved in the
    audit), so the model must never see them.
    """
    if not isinstance(raw, str):
        raise BadURL('not a string')
    text = raw.strip()
    if not text or len(text) > 4096 or re.search(r'[\x00-\x1f\x7f\s]', text):
        raise BadURL('empty/too long/whitespace')
    m = _SCHEME.match(text)
    if m and not text.lower().startswith(('http://', 'https://')):
        raise BadURL('non-http scheme')
    if not m:
        text = 'http://' + text
    text = text.replace('\\', '/')
    try:
        p = urlsplit(text)
        host = (p.hostname or '').lower().rstrip('.')
        port = p.port
    except ValueError as e:
        raise BadURL('unparseable') from None
    if not host or ' ' in host:
        raise BadURL('no host')
    userinfo = p.netloc.rsplit('@', 1)[0] if '@' in p.netloc else ''
    if host.startswith('www.') and host.count('.') >= 2:
        host = host[4:]
    path = p.path or ''
    if path == '/':
        path = ''
    query = '&'.join(kv for kv in (p.query or '').split('&') if kv and not _TRACKING.match(kv.split('=', 1)[0].lower()))
    try:
        ip = ipaddress.ip_address(host.strip('[]'))
        is_ip = True
    except ValueError:
        ip = None
        is_ip = False
    if is_ip:
        reg, sub, suffix, sld = 'ip:' + host, '', '', ''
    else:
        e = _ext(host)
        sld, suffix, sub = e.domain or '', e.suffix or '', e.subdomain or ''
        reg = f'{sld}.{suffix}' if sld and suffix else host
    if port in (80, 443):
        port = None
    platform = None
    for cand in (reg, '.'.join(host.split('.')[-3:])):
        if cand in PLATFORMS:
            platform = cand
    group = host if platform else reg          # unit of grouping / independence
    canon = (userinfo + '@' if userinfo else '') + host + (f':{port}' if port else '') + path + ('?' + query if query else '')
    return dict(canon=canon, host=host, path=path, query=query, reg=reg, sld=sld,
                suffix=suffix, sub=sub, is_ip=is_ip, port=port, userinfo=userinfo,
                platform=platform, group=group)


def shape(p: dict) -> str:
    if not p['path'] and not p['query']:
        return 'root'
    depth = len([s for s in p['path'].split('/') if s])
    return 'shallow' if depth <= 1 and not p['query'] else 'deep'


def _entropy(s: str) -> float:
    if not s:
        return 0.0
    c = Counter(s); n = len(s)
    return -sum(v / n * math.log2(v / n) for v in c.values())


def _brand_hits(text: str, exclude: str) -> int:
    toks = set(re.findall(r'[a-z0-9]+', text))
    hits = {t for t in toks if t in BRANDS and (len(t) >= 4 or t in _BRAND_SHORT_OK)}
    # embedded brand substrings in longer tokens (e.g. paypal-secure, appleid)
    for t in toks:
        if len(t) > 6:
            for b in _BRAND_MATCH:
                if b in t and b != t:
                    hits.add(b)
    hits.discard(exclude)
    return len(hits)


NUM_FEATURES = [
    'host_len', 'n_labels', 'sub_labels', 'max_label_len', 'host_digits', 'host_hyphens',
    'host_entropy', 'sld_len', 'sld_digits', 'sld_hyphens', 'suffix_len', 'suffix_is_cc',
    'is_ip', 'punycode', 'platform', 'shortener', 'has_port', 'userinfo',
    'path_len', 'path_depth', 'query_len', 'n_params', 'file_ext', 'path_digits_ratio',
    'path_entropy', 'max_token_len', 'pct_count', 'embedded_url', 'double_slash',
    'brand_in_sub', 'brand_in_path', 'brand_in_sld_mixed', 'lure_words_host', 'lure_words_path',
    'tilde', 'at_in_path', 'is_root', 'is_shallow',
]


def numeric(p: dict) -> list[float]:
    host, path, query, sld, sub, suffix = p['host'], p['path'], p['query'], p['sld'], p['sub'], p['suffix']
    labels = host.split('.')
    pq = (path + '?' + query).lower()
    toks_path = re.findall(r'[a-z0-9]+', pq)
    depth = len([s for s in path.split('/') if s])
    sh = shape(p)
    sld_brand_mixed = 0.0
    for b in _BRAND_MATCH:
        if b in sld and sld != b:
            sld_brand_mixed = 1.0; break
    return [
        len(host), len(labels), len(sub.split('.')) if sub else 0, max(map(len, labels)),
        sum(c.isdigit() for c in host), host.count('-'), _entropy(host),
        len(sld), sum(c.isdigit() for c in sld), sld.count('-'), len(suffix),
        float(len(suffix.split('.')[-1]) == 2), float(p['is_ip']), float('xn--' in host),
        float(p['platform'] is not None), float(p['reg'] in SHORTENERS or host in SHORTENERS),
        float(p['port'] is not None), float(bool(p['userinfo'])),
        len(path), depth, len(query), len(parse_qsl(query, keep_blank_values=True)),
        float(bool(FILE_EXT.search(path))), sum(c.isdigit() for c in pq) / max(len(pq), 1),
        _entropy(pq), max(map(len, toks_path)) if toks_path else 0, pq.count('%'),
        float(bool(re.search(r'https?(:|%3a)', pq))), float('//' in path),
        float(_brand_hits(sub, sld) > 0), float(_brand_hits(pq, sld) > 0), sld_brand_mixed,
        sum(t in LURE_WORDS for t in re.findall(r'[a-z]+', sub + '.' + sld)),
        sum(t in LURE_WORDS for t in re.findall(r'[a-z]+', pq)),
        float('~' in path), float('@' in pq), float(sh == 'root'), float(sh == 'shallow'),
    ]


def numeric_matrix(parsed: list[dict]) -> np.ndarray:
    X = np.asarray([numeric(p) for p in parsed], dtype=np.float32)
    assert X.shape[1] == len(NUM_FEATURES)
    return X


def host_text(p: dict) -> str:
    # TLD + labels with boundary markers; sld kept so char-ngrams see brand lookalikes.
    return '^' + p['host'] + '$'


def path_text(p: dict) -> str:
    return (p['path'] + ('?' + p['query'] if p['query'] else '')).lower()[:400]


# ---------------- extra features (need small resource files) ----------------
EXTRA_BRANDS = frozenset('''opensea uniswap pancakeswap phantom exodus atomicwallet trezor walletconnect
solana ethereum kucoin bybit okx bitget robinhood revolut wise venmo cashapp zelle klarna
docusign dropbox wetransfer onedrive outlook microsoft office365 netflix disney hbomax
roblox fortnite playstation xbox nintendo telegram whatsapp signal instagram tiktok
paypal amazon apple icloud google youtube facebook linkedin twitter spotify'''.split())
LOOKALIKE_BRANDS = sorted({b for b in (BRANDS | EXTRA_BRANDS) if len(b) >= 5})
_HOMO = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't', '8': 'b', '$': 's'})
EXTRA_FEATURES = ['lookalike_d1', 'lookalike_d2', 'brand_token_in_host', 'lm_sld', 'lm_min3', 'vowel_ratio', 'consonant_run', 'tld_id']


class CharLM:
    """add-k smoothed char trigram LM over '^sld$' strings."""
    def __init__(self, tri=None, bi=None, k=0.1, V=40):
        self.tri, self.bi, self.k, self.V = tri or {}, bi or {}, k, V
    @classmethod
    def fit(cls, words):
        tri, bi = Counter(), Counter()
        for w in words:
            s = '^^' + w + '$'
            for i in range(len(s) - 2):
                tri[s[i:i+3]] += 1; bi[s[i:i+2]] += 1
        return cls(dict(tri), dict(bi))
    def logps(self, w):
        s = '^^' + w + '$'
        return [math.log((self.tri.get(s[i:i+3], 0) + self.k) / (self.bi.get(s[i:i+2], 0) + self.k * self.V)) for i in range(len(s) - 2)]


def extra(p: dict, lm: CharLM, tld_index: dict) -> list[float]:
    from rapidfuzz.distance import Levenshtein
    sld = p['sld']
    toks = set(re.split(r'[-.]', (p['sub'] + '.' + sld).lower())) | {sld}
    toks |= {t.translate(_HOMO) for t in toks}
    toks = {t for t in toks if len(t) >= 4}
    d1 = d2 = 0.0; tokhit = 0.0
    for t in toks:
        for b in LOOKALIKE_BRANDS:
            if abs(len(t) - len(b)) > 2: 
                if b in t and t != b: tokhit = 1.0
                continue
            if t == b:
                if t != sld: tokhit = 1.0
                continue
            d = Levenshtein.distance(t, b, score_cutoff=2)
            if d == 1: d1 = 1.0
            elif d == 2 and len(b) >= 7: d2 = 1.0
            if b in t: tokhit = 1.0
    if sld in LOOKALIKE_BRANDS:      # the brand's own domain is not a lookalike of itself
        d1 = d2 = 0.0
    lp = lm.logps(sld) if sld and not p['is_ip'] else [0.0]
    letters = [c for c in sld if c.isalpha()]
    vow = sum(c in 'aeiou' for c in letters) / max(len(letters), 1)
    runs = re.findall(r'[bcdfghjklmnpqrstvwxz]+', sld)
    tld = (p['suffix'].split('.')[-1] if p['suffix'] else ('<ip>' if p['is_ip'] else '<none>'))
    return [d1, d2, tokhit, float(np.mean(lp)), float(np.min(lp)) if lp else 0.0, vow,
            max(map(len, runs)) if runs else 0, float(tld_index.get(tld, 0))]

In [ ]:
%%writefile mlphil_features.py
"""Featurisation shared by training and inference."""
import numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer
import mlphil_urlcore as U

HOST_HV = HashingVectorizer(analyzer='char', ngram_range=(2, 5), n_features=2**20, alternate_sign=False, norm=None, lowercase=False)
PATH_HV = HashingVectorizer(analyzer='char', ngram_range=(3, 5), n_features=2**20, alternate_sign=False, norm=None, lowercase=False)
WORD_HV = HashingVectorizer(analyzer='word', token_pattern=r'[a-z0-9]{2,}', n_features=2**18, alternate_sign=False, norm=None, lowercase=False)

def parts_from_rows(df):
    cols = ['canon','host','path','query','reg','sld','suffix','sub','is_ip','port','userinfo','platform','group']
    recs = df[cols].to_dict('records')
    for r in recs:
        if r['platform'] is not None and r['platform'] != r['platform']:  # NaN -> None
            r['platform'] = None
        if r['port'] is not None and r['port'] == r['port']:
            r['port'] = int(r['port'])
        else:
            r['port'] = None
        for k in ('path','query','sub','sld','suffix','userinfo'):
            if r[k] is None or r[k] != r[k]: r[k] = ''
    return recs

def raw_blocks(parsed):
    host = HOST_HV.transform([U.host_text(p) for p in parsed])
    pth = PATH_HV.transform([U.path_text(p) for p in parsed])
    wrd = WORD_HV.transform([U.path_text(p) + ' ' + p['sub'].replace('.', ' ') for p in parsed])
    num = U.numeric_matrix(parsed)
    return host.tocsr(), pth.tocsr(), wrd.tocsr(), num

In [ ]:
import importlib, mlphil_urlcore as U, mlphil_features as F
importlib.reload(U); importlib.reload(F)
for s in ['https://www.Google.com/', 'google.com', 'http://paypal-support-help.blogspot.com/',
          'HTTPS://login.microsoftonline.com.verify-acc.xyz/a/b.php?x=1']:
    p = U.parse(s); print(f'{s:65s} -> canon={p["canon"]:45s} group={p["group"]:35s} shape={U.shape(p)}')

## 4. Exploratory data analysis (raw training data)

In [ ]:
raw = pd.read_json(P_HF)
raw = raw.rename(columns={'text': 'url'})[['url', 'label']]
if CFG['HF_FRACTION'] < 1: raw = raw.sample(frac=CFG['HF_FRACTION'], random_state=SEED).reset_index(drop=True)
print('rows:', len(raw), '| label counts:', raw.label.value_counts().to_dict())
print('exact duplicate URLs:', raw.url.duplicated().sum(),
      '| URLs with conflicting labels:', (raw.groupby('url').label.nunique() > 1).sum())
raw.sample(min(8, len(raw)), random_state=1)

In [ ]:
# Format "shortcut" audit: features of how the URL was WRITTEN, not of the site itself.
def fmt(u):
    t = re.sub(r'^[a-z]+://', '', u, flags=re.I); host = t.split('/')[0].split('?')[0]; rest = t[len(host):]
    return pd.Series(dict(has_scheme=bool(re.match(r'^[a-z][a-z0-9+.-]*://', u, re.I)),
                          https=u.lower().startswith('https://'), www=t.lower().startswith('www.'),
                          root_only=rest in ('', '/'), tld=host.rsplit('.', 1)[-1].lower(),
                          length=len(u), depth=len([s for s in rest.split('?')[0].split('/') if s])))
smp = raw.sample(min(200_000, len(raw)), random_state=SEED)
fx = pd.concat([smp.reset_index(drop=True), smp.url.apply(fmt).reset_index(drop=True)], axis=1)
audit = fx.groupby('label')[['has_scheme', 'https', 'www', 'root_only']].mean().T
audit.columns = ['benign', 'phishing']; audit['ratio'] = audit.phishing / audit.benign
display(audit.round(3))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
audit[['benign', 'phishing']].plot.bar(ax=ax[0], rot=0, title='URL-format artefacts by label')
top_tld = fx.tld.value_counts().head(20).index
(fx[fx.tld.isin(top_tld)].groupby('tld').label.mean().sort_values()
   .plot.barh(ax=ax[1], title='Phishing share by TLD (top-20 TLDs)'))
ax[1].axvline(fx.label.mean(), ls='--', c='k')
for lab, g in fx.groupby('label'):
    ax[2].hist(np.log10(g.length.clip(1)), bins=60, alpha=.55, label=['benign', 'phishing'][lab], density=True)
ax[2].set(title='URL length (log10 chars)'); ax[2].legend()
plt.tight_layout(); plt.savefig(FIG / 'eda_shortcuts.png'); plt.show()
print('Finding: root-only URLs and country TLDs are label-correlated by HOW the data was collected '
      '-> a naive model learns "homepage = phishing" and ".cn/.br/.ru = phishing".')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
pd.crosstab(fx.depth.clip(upper=8), fx.label, normalize='columns').plot.bar(ax=ax[0], rot=0, title='Path depth by label')
ax[0].legend(['benign', 'phishing'])
raw.label.value_counts().rename({0: 'benign', 1: 'phishing'}).plot.pie(ax=ax[1], autopct='%.1f%%', title='Class balance'); ax[1].set_ylabel('')
plt.tight_layout(); plt.show()
del fx, smp; gc.collect()

## 5. Preprocessing
1. Parse + canonicalise every URL; drop non-HTTP / unparseable / non-public hosts (localhost, private IPs).
2. De-duplicate on canonical form; drop canonical URLs with conflicting labels.
3. Build **external cohorts**, then remove every external row whose domain group appears in the training data.
4. Add TLD-balanced Majestic homepages, modern phishing (Phishing.Database rows *not* in external cohorts) and PyPI links to the training pool.
5. Group split the pool 70 / 10 / 20 (train / val / internal test) by domain group.

In [ ]:
FIELDS = ['canon', 'host', 'path', 'query', 'reg', 'sld', 'suffix', 'sub', 'is_ip', 'port', 'userinfo', 'platform', 'group']
def nonpublic(h):
    if h == 'localhost' or h.endswith(('.local', '.localhost')): return True
    try: return not ipaddress.ip_address(h.strip('[]')).is_global
    except ValueError: return False

def to_rows(urls, label, source, quality):
    out, bad = [], 0
    for u in urls:
        try: p = U.parse(u)
        except U.BadURL: bad += 1; continue
        if nonpublic(p['host']): bad += 1; continue
        r = {k: p[k] for k in FIELDS}; r.update(label=label, source=source, label_quality=quality, shape=U.shape(p), raw=u)
        out.append(r)
    df = pd.DataFrame(out); print(f'{source:28s} parsed={len(df):>9,}  rejected={bad:,}')
    return df

def h01(s):  # stable hash -> [0,1)
    return int(hashlib.sha256(s.encode()).hexdigest()[:12], 16) / 16**12

LOG = {}
hf = pd.concat([to_rows(raw.url[raw.label == l], l, 'HF', 'dataset') for l in (0, 1)], ignore_index=True)
conf = hf.groupby('canon').label.nunique(); conf = set(conf[conf > 1].index)
hf = hf[~hf.canon.isin(conf)].drop_duplicates('canon').reset_index(drop=True)
LOG.update(hf_conflicts_dropped=len(conf), hf_rows=len(hf))
G_HF = set(hf.group)
del raw; gc.collect(); print(LOG)

In [ ]:
# ---------- external cohorts ----------
def read_lines(p): return [l.strip() for l in open(p, encoding='utf8', errors='ignore') if l.strip()]
pdb_urls = [u for u in read_lines(P_PDB) if u.lower().startswith(('http://', 'https://'))]
cur_urls = []
for f in sorted(cur_dir.glob('*')):
    cur_urls += [m.rstrip('.,;:*_') for m in re.findall(URL_RX, f.read_text(encoding='utf8', errors='ignore'))]
cur_urls = [u for u in cur_urls if not re.search(r'shields\.io|badge|\.(png|svg|gif|jpe?g)(\?|$)', u, re.I)]

if str(P_TRA).endswith('.zip'):
    z = zipfile.ZipFile(P_TRA); tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
else:
    tranco = pd.read_csv(P_TRA, header=None, names=['rank', 'domain'])
tranco = tranco.sort_values('rank').head(CFG['TRANCO_TOP'])

pdb_all = to_rows(pdb_urls, 1, 'PhishingDatabase_ACTIVE', 'feed_listed')
ext = pd.concat([to_rows(read_lines(P_OPF), 1, 'OpenPhish_feed', 'feed_listed'), pdb_all,
                 to_rows('https://' + tranco.domain.astype(str) + '/', 0, 'Tranco_roots', 'popularity_proxy'),
                 to_rows(cur_urls, 0, 'Curated_link_lists', 'human_curated_proxy')], ignore_index=True)
ext = ext.drop_duplicates(['canon', 'source'])
phish_canon = set(ext[ext.label == 1].canon) | set(hf[hf.label == 1].canon)
ext = ext[~((ext.label == 0) & ext.canon.isin(phish_canon))]           # a URL listed as phishing is never "benign"
before = ext.source.value_counts()
ext = ext[~ext.group.isin(G_HF)]                                         # domain-disjoint from training data
ext = ext.sample(frac=1, random_state=7).groupby(['source', 'group']).head(3)   # max 3 rows per domain
ext['split'] = ext.group.map(lambda g: 'ext_dev' if h01('e|' + g) < .5 else 'ext_test')
parts = [g.sample(CFG['PDB_TEST_ROWS_PER_HALF'], random_state=3) if s == 'PhishingDatabase_ACTIVE' and len(g) > CFG['PDB_TEST_ROWS_PER_HALF'] else g
         for (s, sp_), g in ext.groupby(['source', 'split'])]
ext = pd.concat(parts, ignore_index=True)
G_EXT = set(ext.group)
display(pd.DataFrame({'before_disjoint': before, 'after_disjoint+cap': ext.source.value_counts()}))

In [ ]:
# ---------- training-pool additions ----------
# (a) Majestic homepages, TLD-balanced: for each TLD add benign roots up to the phishing surplus (floor 50)
maj = pd.read_csv(P_MAJ, usecols=['Domain'])
pdb_hosts = set(pdb_all.host) | set(pdb_all.reg)
mj = to_rows(maj.Domain.astype(str), 0, 'Majestic_roots', 'popularity_proxy')
phish_groups = set(hf[hf.label == 1].group) | set(ext[ext.label == 1].group)
mj = mj[(mj['shape'] == 'root') & mj.platform.isna() & ~mj.reg.isin(U.SHORTENERS) & ~mj.group.isin(G_EXT | phish_groups | G_HF)
        & ~mj.host.isin(pdb_hosts) & ~mj.reg.isin(pdb_hosts)]
tld = lambda s: s.fillna('').str.split('.').str[-1]
hf['tld'] = tld(hf.suffix); mj['tld'] = tld(mj.suffix)
surplus = (hf[hf.label == 1].tld.value_counts() - hf[hf.label == 0].tld.value_counts()).fillna(hf[hf.label == 1].tld.value_counts()).clip(lower=0)
picks = []
for t, g in mj.groupby('tld'):
    k = max(int(min(len(g), surplus.get(t, 0))), min(len(g), 50))
    picks.append(g.sample(k, random_state=11) if k < len(g) else g)
mj = pd.concat(picks); mj = mj.sample(min(len(mj), CFG['MAJESTIC_CAP']), random_state=11)

# (b) modern phishing: Phishing.Database rows whose domain group is NOT in any external cohort
pdb_tr = pdb_all[~pdb_all.group.isin(G_EXT) & ~pdb_all.canon.isin(set(hf.canon))]
pdb_tr = pdb_tr.sample(frac=1, random_state=5).groupby('group').head(2)
pdb_tr = pdb_tr.sample(min(len(pdb_tr), CFG['PDB_TRAIN_ROWS']), random_state=5).assign(source='PhishingDatabase_train')

# (c) modern benign deep links from PyPI READMEs
py = to_rows(read_lines(P_PYPI), 0, 'PyPI_readme_links', 'maintainer_published_proxy')
py = py[~py.group.isin(G_EXT) & ~py.canon.isin(phish_canon) & ~py.reg.isin({'example.com', 'example.org', 'example.net'}) & ~py.is_ip]
py = py.drop_duplicates('canon').sample(frac=1, random_state=5).groupby('group').head(5)

pool = pd.concat([hf, mj, pdb_tr, py], ignore_index=True)
assert (pool.groupby('canon').label.nunique() == 1).all(), 'same URL with two labels'
pool = pool.drop_duplicates('canon').reset_index(drop=True)   # same URL from two sources -> keep first (HF > Majestic > PDB > PyPI)
pool['tld'] = tld(pool.suffix); ext['tld'] = tld(ext.suffix)
pool['split'] = pool.group.map(lambda g: 'train' if h01('s|' + g) < .70 else ('val' if h01('s|' + g) < .80 else 'test'))
pool = pool.drop(columns=['raw', 'label_quality'])
del maj, mj, pdb_all, pdb_tr, py, hf, pdb_urls, cur_urls; gc.collect()

# ---------- leakage checks (hard asserts) ----------
gs = {s: set(pool.group[pool.split == s]) for s in ('train', 'val', 'test')}
assert not (gs['train'] & gs['val']) and not (gs['train'] & gs['test']) and not (gs['val'] & gs['test'])
assert not (set(pool.group) & G_EXT), 'external cohort domain leaked into training pool'
assert not (set(ext.group[ext.split == 'ext_dev']) & set(ext.group[ext.split == 'ext_test']))
assert not pool.canon.duplicated().any()
print('leakage checks passed: 0 shared domain groups across train/val/test/ext_dev/ext_test')
display(pd.crosstab([pool.split, pool.source], pool.label, margins=True))
display(pd.crosstab([ext.split, ext.source], ext.label))

In [ ]:
# EDA after preprocessing: did TLD / shape bias shrink?
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
t = pool[pool.split == 'train']
chk = ['com', 'org', 'net', 'ru', 'cn', 'br', 'jp', 'de', 'in', 'fr', 'io', 'uk', 'xyz', 'top']
t[t.tld.isin(chk)].groupby('tld').label.mean().reindex(chk).plot.bar(ax=ax[0], rot=0, title='Phishing share by TLD — training pool after balancing')
pd.crosstab(t['shape'], t.label, normalize='index').plot.bar(ax=ax[1], rot=0, stacked=True, title='Label mix by URL shape (before sample weights)')
ax[1].legend(['benign', 'phishing']); plt.tight_layout(); plt.show()
print('Remaining shape imbalance is removed with per-(shape,label) sample weights during training.')

## 6. Feature engineering
* **Text blocks:** char 2–5-grams of the host, char 3–5-grams of the path, word tokens (hashing → TF-IDF).
* **46 structural features:** lengths, entropy, IP host, punycode, shared platform, shortener, brand-in-subdomain, brand look-alike (edit distance + homoglyphs), lure words, char-LM "naturalness" of the domain, TLD id, …

In [ ]:
# char-trigram LM of normal domain names: fitted on Majestic names NOT used anywhere in pool/external (no leakage)
used = set(pool.group) | G_EXT
slds = []
for d in pd.read_csv(P_MAJ, usecols=['Domain']).Domain.astype(str):
    e = U._EXT(d.lower())
    if e.domain and e.suffix and f'{e.domain}.{e.suffix}' not in used: slds.append(e.domain)
LM = U.CharLM.fit(slds); del slds
TLD_INDEX = {t: i + 1 for i, t in enumerate(pool[pool.split == 'train'].tld.value_counts().head(400).index)}

def featurise(df, chunk=150_000):
    parts = {'host': [], 'path': [], 'word': [], 'tab': []}
    for i in range(0, len(df), chunk):              # chunked -> lower peak memory
        parsed = F.parts_from_rows(df.iloc[i:i + chunk])
        h, p, w, num = F.raw_blocks(parsed)
        extra = np.asarray([U.extra(q, LM, TLD_INDEX) for q in parsed], dtype=np.float32)
        for k, v in (('host', h), ('path', p), ('word', w)): parts[k].append(v.astype(np.float32))
        parts['tab'].append(np.hstack([num, extra]).astype(np.float32))
        del parsed; gc.collect()
    return {k: (sp.vstack(v, format='csr') if k != 'tab' else np.vstack(v)) for k, v in parts.items()}
t1 = time.time()
pool = pool.reset_index(drop=True); ext = ext.reset_index(drop=True)
BP = featurise(pool); BE = featurise(ext)
TAB_FEATS = U.NUM_FEATURES + U.EXTRA_FEATURES
print(f'featurised {len(pool):,} + {len(ext):,} rows in {time.time()-t1:.0f}s; tabular features = {len(TAB_FEATS)}')

In [ ]:
tab = pd.DataFrame(BP['tab'][pool.split.eq('train').values], columns=TAB_FEATS).assign(label=pool.label[pool.split.eq('train')].values)
show = ['is_ip', 'platform', 'brand_in_sub', 'brand_in_path', 'lookalike_d1', 'brand_token_in_host', 'punycode', 'embedded_url', 'file_ext', 'userinfo']
display(tab.groupby('label')[show].mean().T.rename(columns={0: 'benign', 1: 'phishing'}).round(4))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for i, f in enumerate(['lm_sld', 'host_entropy', 'path_len']):
    for lab in (0, 1):
        v = tab.loc[tab.label == lab, f]; ax[i].hist(v.clip(v.quantile(.005), v.quantile(.995)), bins=60, alpha=.55, density=True, label=['benign', 'phishing'][lab])
    ax[i].set_title(f); ax[i].legend()
plt.tight_layout(); plt.show(); del tab

## 7. Model training
Two-stage stack:
1. **Text models** (logistic regression via SGD): one on the host, one on the path. Their scores for training rows are **out-of-fold** (grouped folds) so stage 2 never sees self-predictions.
2. **LightGBM** on 46 structural features + 2 text scores, early-stopped on the validation split.

Sample weights equalise every (shape × label) cell → the model cannot use "homepage vs deep link" as a label shortcut.

In [ ]:
y = pool.label.values
TR = pool.split.eq('train').values; VA = pool.split.eq('val').values
W = np.zeros(len(pool))
for s in ('root', 'shallow', 'deep'):
    for lab in (0, 1):
        m = TR & pool['shape'].eq(s).values & (y == lab); W[m] = TR.sum() / (6 * max(m.sum(), 1))
W[TR & pool.source.eq('PyPI_readme_links').values] *= CFG['PYPI_WEIGHT']
W[TR] *= TR.sum() / W[TR].sum()
display(pd.DataFrame({'rows': pool[TR].groupby(['shape', 'label']).size(),
                      'total_weight': pd.Series(W[TR]).groupby([pool['shape'][TR].values, y[TR]]).sum().round(0)}))

TEXT = {'host': ('host',), 'path': ('path', 'word')}
def fit_text(kind, idx):
    tfs = {n: TfidfTransformer(sublinear_tf=True).fit(BP[n][idx]) for n in TEXT[kind]}
    X = sp.hstack([normalize(tfs[n].transform(BP[n][idx])) for n in TEXT[kind]], format='csr')
    clf = SGDClassifier(loss='log_loss', alpha=CFG['SGD_ALPHA'], max_iter=25, tol=1e-5, random_state=0, average=True)
    return tfs, clf.fit(X, y[idx], sample_weight=W[idx])
def apply_text(kind, model, B, idx):
    tfs, clf = model; out = np.empty(len(idx))
    for i in range(0, len(idx), 100_000):
        j = idx[i:i + 100_000]
        out[i:i + len(j)] = clf.decision_function(sp.hstack([normalize(tfs[n].transform(B[n][j])) for n in TEXT[kind]], format='csr'))
    return out

t1 = time.time()
fold = pool.group.map(lambda g: int(hashlib.sha256(('f|' + g).encode()).hexdigest()[:8], 16) % CFG['TEXT_FOLDS']).values
tr_idx, ntr_idx = np.where(TR)[0], np.where(~TR)[0]
SP, SE = np.zeros((len(pool), 2)), np.zeros((len(ext), 2)); TEXT_MODELS = {}
for k, kind in enumerate(TEXT):
    for f in range(CFG['TEXT_FOLDS']):
        fi, fo = tr_idx[fold[tr_idx] != f], tr_idx[fold[tr_idx] == f]
        SP[fo, k] = apply_text(kind, fit_text(kind, fi), BP, fo)
    TEXT_MODELS[kind] = m = fit_text(kind, tr_idx)
    SP[ntr_idx, k] = apply_text(kind, m, BP, ntr_idx); SE[:, k] = apply_text(kind, m, BE, np.arange(len(ext)))
    print(f'text model [{kind}] done  {time.time()-t1:.0f}s')
FEATS = TAB_FEATS + ['s_host', 's_path']
XP = np.hstack([BP['tab'], SP]).astype(np.float32); XE = np.hstack([BE['tab'], SE]).astype(np.float32)
del BP, BE; gc.collect()   # sparse text blocks no longer needed

In [ ]:
CAT = [FEATS.index('tld_id')]
GBM_PARAMS = dict(objective='binary', learning_rate=CFG['GBM_LR'], num_leaves=CFG['GBM_LEAVES'], min_data_in_leaf=CFG['GBM_MIN_CHILD'],
                  feature_fraction=.8, bagging_fraction=.8, bagging_freq=1, lambda_l2=1.0, cat_smooth=50,
                  min_data_per_group=200, max_cat_to_onehot=4, verbose=-1, num_threads=os.cpu_count(), seed=0)
def train_gbm(seed=0, rounds=None, idx_tr=None, idx_va=None):
    idx_tr = np.where(TR)[0] if idx_tr is None else idx_tr
    p = dict(GBM_PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
    dtr = lgb.Dataset(XP[idx_tr], y[idx_tr], weight=W[idx_tr], feature_name=FEATS, categorical_feature=CAT, free_raw_data=False)
    if rounds:   # fixed number of rounds (stability re-runs)
        return lgb.train(p, dtr, rounds)
    idx_va = np.where(VA)[0] if idx_va is None else idx_va
    dva = lgb.Dataset(XP[idx_va], y[idx_va], reference=dtr, categorical_feature=CAT)
    return lgb.train(p, dtr, CFG['GBM_MAX_ROUNDS'], valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)])

t1 = time.time(); GBM = train_gbm()
BEST = GBM.best_iteration; print(f'LightGBM best iteration {BEST}  ({time.time()-t1:.0f}s)')
pool['score'] = GBM.predict(XP, num_iteration=BEST); ext['score'] = GBM.predict(XE, num_iteration=BEST)
imp = pd.Series(GBM.feature_importance('gain'), index=FEATS); imp = (imp / imp.sum()).sort_values()
imp.tail(20).plot.barh(figsize=(7, 6), title='LightGBM feature importance (gain share)'); plt.tight_layout(); plt.savefig(FIG / 'importance.png'); plt.show()

## 8. Threshold selection (dev data only)
* **HIGH** = lowest score threshold whose false-alert rate is ≤ 3 % on **every** benign dev cohort (internal val, Tranco dev, curated dev).
* **MEDIUM** = same rule at ≤ 10 %. Below MEDIUM = LOW.

In [ ]:
def thr_for_fpr(score_sets, max_fpr):
    t = 0.0
    for s in score_sets:
        s = np.sort(np.asarray(s)); k = int(np.floor(max_fpr * len(s)))
        c = s[len(s) - k] if k > 0 else np.nextafter(s[-1], 1)
        while (s >= c).mean() > max_fpr: c = np.nextafter(c, 1)
        t = max(t, float(c))
    return t
def dev_benign(df_pool, df_ext, col='score'):
    return [df_pool.loc[df_pool.split.eq('val') & df_pool.label.eq(0), col].values,
            df_ext.loc[df_ext.split.eq('ext_dev') & df_ext.source.eq('Tranco_roots'), col].values,
            df_ext.loc[df_ext.split.eq('ext_dev') & df_ext.source.eq('Curated_link_lists'), col].values]
THR_HIGH = thr_for_fpr(dev_benign(pool, ext), CFG['ALERT_MAX_FPR'])
THR_MED = thr_for_fpr(dev_benign(pool, ext), CFG['REVIEW_MAX_FPR'])
print(f'HIGH >= {THR_HIGH:.4f}   MEDIUM >= {THR_MED:.4f}')

## 9. Evaluation
Internal (grouped) and external (domain-disjoint) cohorts. 95 % Wilson confidence intervals. Test cohorts are read here for the first time.

In [ ]:
def wilson(k, n): return proportion_confint(k, n, method='wilson') if n else (np.nan, np.nan)
def cohort_table(df_pool, df_ext, thr_hi=None, thr_med=None, col='score'):
    thr_hi = THR_HIGH if thr_hi is None else thr_hi; thr_med = THR_MED if thr_med is None else thr_med
    allr = pd.concat([df_pool[df_pool.split.isin(['val', 'test'])][['split', 'source', 'label', col]],
                      df_ext[['split', 'source', 'label', col]]])
    rows = []
    def add(split, src, g):
        k, n = int((g[col] >= thr_hi).sum()), len(g); lo, hi = wilson(k, n); lab = int(g.label.iloc[0])
        rows.append(dict(split=split, cohort=src, label='phishing' if lab else 'benign', n=n,
                         metric='recall' if lab else 'false_alert', rate_HIGH=k / n, ci_lo=lo, ci_hi=hi,
                         rate_HIGH_or_MED=float((g[col] >= thr_med).mean())))
    for (s, src, l), g in allr.groupby(['split', 'source', 'label']): add(s, src, g)
    for s in ('val', 'test'):
        for l, name in ((0, 'ALL_benign'), (1, 'ALL_phishing')):
            add(s, name, allr[(allr.split == s) & (allr.label == l)])
    order = {'val': 0, 'test': 1, 'ext_dev': 2, 'ext_test': 3}
    return pd.DataFrame(rows).sort_values(['split', 'label'], key=lambda c: c.map(order) if c.name == 'split' else c).reset_index(drop=True)
RESULTS = cohort_table(pool, ext)
RESULTS.to_csv(WORK / 'results_by_cohort.csv', index=False)
RESULTS.round(4)

In [ ]:
te = pool[pool.split == 'test']
fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
cm = confusion_matrix(te.label, te.score >= THR_HIGH)
ax[0].imshow(cm, cmap='Blues'); ax[0].set(title='Internal test @HIGH', xticks=[0, 1], yticks=[0, 1], xticklabels=['pred benign', 'pred phish'], yticklabels=['benign', 'phishing'])
for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, f'{v:,}', ha='center', va='center', color='red')
ax[0].grid(False)
ext_t = ext[ext.split == 'ext_test']
for name, d in (('internal test', te), ('external test', ext_t)):
    fpr, tpr, _ = roc_curve(d.label, d.score); pr, rc, _ = precision_recall_curve(d.label, d.score)
    ax[1].plot(fpr, tpr, label=f'{name} AUC={roc_auc_score(d.label, d.score):.3f}')
    ax[2].plot(rc, pr, label=f'{name} AP={average_precision_score(d.label, d.score):.3f}')
ax[1].set(xscale='log', xlim=(1e-3, 1), title='ROC (log FPR)', xlabel='false alert rate', ylabel='recall'); ax[1].legend()
ax[2].set(title='Precision-recall', xlabel='recall', ylabel='precision'); ax[2].legend()
for (src, lab), g in ext_t.groupby(['source', 'label']):
    ax[3].hist(g.score, bins=50, histtype='step', lw=1.6, density=True, label=src)
ax[3].axvline(THR_HIGH, c='r', ls='--'); ax[3].axvline(THR_MED, c='orange', ls='--')
ax[3].set(yscale='log', title='Score distribution by external cohort'); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / 'evaluation.png'); plt.show()

In [ ]:
# Breakdowns on the external test: URL shape and TLD
bt = ext_t.assign(alert=ext_t.score >= THR_HIGH)
display(bt.pivot_table(index='source', columns='shape', values='alert', aggfunc='mean').round(3))
top = bt.tld.value_counts().head(12).index
display(bt[bt.tld.isin(top)].pivot_table(index='tld', columns='source', values='alert', aggfunc='mean').round(3))

# calibration (internal test)
pt, pp = calibration_curve(te.label, te.score, n_bins=15, strategy='quantile')
plt.figure(figsize=(4.5, 4)); plt.plot(pp, pt, 'o-'); plt.plot([0, 1], [0, 1], 'k--')
plt.title('Calibration — internal test'); plt.xlabel('mean predicted'); plt.ylabel('observed phishing share'); plt.show()
print('Note: weighting balances classes, so scores are a RANKING / risk signal, not a calibrated probability.')

In [ ]:
def lookup(split, cohort): return RESULTS[(RESULTS.split == split) & (RESULTS.cohort == cohort)].iloc[0].rate_HIGH
CHECK = {}
for name, (split, cohort, op, target) in CRITERIA.items():
    v = lookup(split, cohort); CHECK[name] = dict(value=round(v, 4), target=target, passed=bool(v >= target if op == '>=' else v <= target))
CHECK = pd.DataFrame(CHECK).T
benign_ok = CHECK.loc[[k for k in CHECK.index if 'false alert' in k], 'passed'].all()
VERDICT = 'ADVISORY_READY' if CHECK.passed.all() else ('ADVISORY_WITH_CAVEATS' if benign_ok else 'RESEARCH_ONLY')
display(CHECK); print('VERDICT:', VERDICT, '(URL-only model = advisory risk signal, never a safety guarantee)')

## 10. Model stability tests
| Test | Question |
|---|---|
| 10.1 Canonical invariance | Do cosmetic rewrites of the same URL (scheme, `www.`, case, trailing `/`, default port, fragment, tracking params) change the score? (must be **0**) |
| 10.2 Perturbation sensitivity | How much do harmless but *non-removable* edits (generic query parameter) move the score / flip the band? |
| 10.3 Seed stability | Retrain LightGBM with different random seeds — how much do test metrics move? |
| 10.4 Grouped cross-validation | Is performance consistent across disjoint domain folds? |
| 10.5 Cluster bootstrap | Confidence intervals that respect domain clustering |
| 10.6 Threshold sensitivity | How fragile are metrics to small threshold changes? |
| 10.7 Dev → test consistency | Did dev results transfer to unseen test cohorts? |

In [ ]:
def score_urls(urls):
    """score raw URL strings with the trained stack (same path as the exported predictor)"""
    parsed, ok = [], []
    for u in urls:
        try: parsed.append(U.parse(u)); ok.append(True)
        except U.BadURL: ok.append(False)
    out = np.full(len(urls), np.nan)
    if parsed:
        h, p, w, num = F.raw_blocks(parsed)
        B = {'host': h.astype(np.float32), 'path': p.astype(np.float32), 'word': w.astype(np.float32)}
        extra = np.asarray([U.extra(q, LM, TLD_INDEX) for q in parsed], dtype=np.float32)
        s = np.column_stack([apply_text(k, TEXT_MODELS[k], B, np.arange(len(parsed))) for k in TEXT])
        out[np.where(ok)[0]] = GBM.predict(np.hstack([num, extra, s]), num_iteration=BEST)
    return out

# 10.1 canonical invariance
base = ext[~ext.userinfo.astype(bool) & ext.port.isna() & ~ext.is_ip].sample(400, random_state=1).canon.tolist()
def variants(c):
    host = U.parse(c)['host']; rest = c[len(host):]          # canon = host + path/query (no userinfo/port here)
    return {'http://': 'http://' + c, 'https://': 'https://' + c, 'https://www.': 'https://www.' + c,
            'UPPER host': 'https://' + host.upper() + rest, 'trailing / on homepage': 'https://' + host + (rest or '/'),
            ':443 default port': 'https://' + host + ':443' + rest, '#fragment': 'https://' + c + '#top',
            '+ utm_source tracking param': 'https://' + c + ('&' if '?' in rest else '?') + 'utm_source=newsletter'}
s0 = score_urls(base); inv = {}
for name in variants(base[0]):
    sv = score_urls([variants(c)[name] for c in base]); inv[name] = np.nanmax(np.abs(sv - s0))
INV = pd.Series(inv, name='max |Δscore|'); display(INV.to_frame())
print('canonical invariance:', 'PASS' if INV.max() < 1e-9 else 'FAIL')

In [ ]:
# 10.2 perturbation sensitivity (harmless edits that are NOT canonicalised away)
def band(s): return np.where(s >= THR_HIGH, 2, np.where(s >= THR_MED, 1, 0))
samp = ext.sample(min(1500, len(ext)), random_state=2)
b0 = score_urls(samp.raw.tolist())
add = lambda u, kv: u.split('#')[0] + ('&' if '?' in u else '?') + kv
pert = {'+ ?lang=en': [add(u, 'lang=en') for u in samp.raw],
        '+ ?ref=newsletter': [add(u, 'ref=newsletter') for u in samp.raw],
        '+ ?id=12345': [add(u, 'id=12345') for u in samp.raw]}
lab = samp.label.values; rows = []
for name, us in pert.items():
    s = score_urls(us); d = np.abs(s - b0); b1 = band(s); bb = band(b0)
    rows.append({'perturbation': name, 'mean |Δ|': np.nanmean(d), 'p95 |Δ|': np.nanpercentile(d, 95),
                 'band flip rate': np.mean(b1 != bb),
                 'benign pushed up a band': np.mean(b1[lab == 0] > bb[lab == 0]),
                 'phishing pushed down a band': np.mean(b1[lab == 1] < bb[lab == 1])})
PERT = pd.DataFrame(rows); display(PERT.round(4))
print('Interpretation: a URL with a query string is a different URL, so some movement is expected; large flips = the model leans on "has query" as a cue.')

In [ ]:
# 10.3 seed stability: retrain LightGBM (fixed #rounds) with new seeds; threshold re-selected per seed by the SAME dev rule
seed_rows = []
for sd in range(1, CFG['STABILITY_SEEDS'] + 1):
    g = train_gbm(seed=sd, rounds=BEST)
    p_ = pool[['split', 'source', 'label']].assign(score=g.predict(XP)); e_ = ext[['split', 'source', 'label']].assign(score=g.predict(XE))
    th = thr_for_fpr(dev_benign(p_, e_), CFG['ALERT_MAX_FPR']); tm = thr_for_fpr(dev_benign(p_, e_), CFG['REVIEW_MAX_FPR'])
    r = cohort_table(p_, e_, th, tm); r = r[r.split.isin(['test', 'ext_test'])]
    seed_rows.append(r.set_index(['split', 'cohort', 'label']).rate_HIGH.rename(f'seed{sd}'))
    print(f'seed {sd} done (thr {th:.4f})')
SEEDS = pd.concat(seed_rows, axis=1)
SEED_SUM = pd.DataFrame({'main_model': RESULTS.set_index(['split', 'cohort', 'label']).rate_HIGH.reindex(SEEDS.index),
                         'seed_mean': SEEDS.mean(1), 'seed_std': SEEDS.std(1), 'seed_min': SEEDS.min(1), 'seed_max': SEEDS.max(1)})
SEED_SUM.round(4)

In [ ]:
# 10.4 grouped K-fold CV (stage-2 model on out-of-fold stack features of the training split)
cv_fold = pool.group.map(lambda g: int(hashlib.sha256(('cv|' + g).encode()).hexdigest()[:8], 16) % CFG['CV_FOLDS']).values
cv_rows = []
for f in range(CFG['CV_FOLDS']):
    it, iv = np.where(TR & (cv_fold != f))[0], np.where(TR & (cv_fold == f))[0]
    g = train_gbm(seed=f, rounds=BEST, idx_tr=it)
    s = g.predict(XP[iv]); yy = y[iv]
    th = np.quantile(s[yy == 0], 1 - CFG['ALERT_MAX_FPR'])
    cv_rows.append(dict(fold=f, n=len(iv), AUC=roc_auc_score(yy, s), AP=average_precision_score(yy, s),
                        recall_at_3pct_FPR=float((s[yy == 1] >= th).mean())))
CV = pd.DataFrame(cv_rows)
display(CV.round(4)); display(CV[['AUC', 'AP', 'recall_at_3pct_FPR']].agg(['mean', 'std']).round(4))

In [ ]:
# 10.5 cluster bootstrap on external test (resample DOMAIN GROUPS, not rows)
rng = np.random.default_rng(SEED); boot = []
for (src, lab), g in ext_t.groupby(['source', 'label']):
    agg = g.assign(a=g.score >= THR_HIGH).groupby('group').agg(k=('a', 'sum'), n=('a', 'size'))
    k, n = agg.k.values, agg.n.values; G = len(agg)
    idx = rng.integers(0, G, size=(CFG['BOOTSTRAP'], G))
    rates = k[idx].sum(1) / n[idx].sum(1)
    boot.append(dict(cohort=src, groups=G, rate=k.sum() / n.sum(), boot_lo=np.percentile(rates, 2.5), boot_hi=np.percentile(rates, 97.5), boot_std=rates.std()))
BOOT = pd.DataFrame(boot); BOOT.round(4)

In [ ]:
# 10.6 threshold sensitivity
grid = np.linspace(max(0, THR_HIGH - .06), min(1, THR_HIGH + .06), 25)
fig, ax = plt.subplots(figsize=(8, 4))
for (src, lab), g in ext_t.groupby(['source', 'label']):
    ax.plot(grid, [(g.score >= t).mean() for t in grid], label=f'{src} ({"recall" if lab else "false alert"})')
ax.axvline(THR_HIGH, c='k', ls='--'); ax.set(title='Alert rate vs threshold (external test)', xlabel='threshold'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / 'threshold_sensitivity.png'); plt.show()
sens = {src: abs((g.score >= THR_HIGH + .01).mean() - (g.score >= THR_HIGH - .01).mean()) for (src, _), g in ext_t.groupby(['source', 'label'])}
print('change in alert rate for ±0.01 threshold shift:', {k: round(v, 4) for k, v in sens.items()})

# 10.7 dev -> test consistency
r = RESULTS[RESULTS.split.isin(['ext_dev', 'ext_test'])].pivot(index='cohort', columns='split', values='rate_HIGH')
r['test - dev'] = r.ext_test - r.ext_dev
r2 = RESULTS[RESULTS.split.isin(['val', 'test'])].pivot(index=['cohort', 'label'], columns='split', values='rate_HIGH'); r2['test - val'] = r2.test - r2.val
display(r.round(4)); display(r2.round(4))

## 11. Export (pickle-free bundle) & parity check
The bundle stores only `.npz`, `.json` and LightGBM text — loading it cannot execute code.

In [ ]:
%%writefile mlphil_predictor.py
"""ML Phil URL predictor — URL-string risk signal only. Never fetches/opens the URL. No pickle."""
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np, scipy.sparse as sp, lightgbm as lgb

class URLRisk:
    def __init__(self, bundle_dir):
        d = Path(bundle_dir); sys.path.insert(0, str(d))
        import mlphil_urlcore as U, mlphil_features as F
        self.U, self.F = U, F
        self.meta = json.loads((d / 'metadata.json').read_text())
        self.A = dict(np.load(d / 'text_models.npz'))
        self.gbm = lgb.Booster(model_file=str(d / 'gbm.txt'))
        r = json.loads((d / 'resources.json').read_text())
        self.lm = U.CharLM(r['lm_tri'], r['lm_bi']); self.tld_index = r['tld_index']
        self.thr_high, self.thr_med = float(self.meta['thr_high']), float(self.meta['thr_medium'])
        assert self.gbm.feature_name() == self.meta['feats'], 'feature schema mismatch'

    def _text(self, kind, blocks):
        mats = []
        for n in self.meta['text_blocks'][kind]:
            M = blocks[n].astype(np.float64).tocsr(copy=True); M.data = 1.0 + np.log(M.data)
            M = M @ sp.diags(self.A[f'{kind}__{n}__idf'].astype(np.float64))
            nr = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()); nr[nr == 0] = 1
            mats.append(sp.diags(1 / nr) @ M)
        return sp.hstack(mats, format='csr') @ self.A[f'{kind}__coef'].astype(np.float64) + self.A[f'{kind}__intercept'][0]

    def score_many(self, urls):
        parsed, ok = [], []
        for u in urls:
            try: parsed.append(self.U.parse(u)); ok.append(True)
            except self.U.BadURL: ok.append(False)
        out = np.full(len(urls), np.nan)
        if parsed:
            h, p, w, num = self.F.raw_blocks(parsed)
            extra = np.asarray([self.U.extra(q, self.lm, self.tld_index) for q in parsed], dtype=np.float32)
            b = {'host': h, 'path': p, 'word': w}
            X = np.hstack([num, extra, self._text('host', b)[:, None], self._text('path', b)[:, None]])
            out[np.where(ok)[0]] = self.gbm.predict(X)
        return out

    def assess(self, url):
        s = float(self.score_many([url])[0])
        if np.isnan(s): return dict(risk='INVALID', score=None, note='Not a valid public http(s) URL string')
        risk = 'HIGH' if s >= self.thr_high else ('MEDIUM' if s >= self.thr_med else 'LOW')
        note = {'HIGH': 'URL pattern strongly resembles phishing', 'MEDIUM': 'Some phishing-like URL patterns; be careful',
                'LOW': 'No strong URL-pattern warning. NOT a safety guarantee.'}[risk]
        return dict(risk=risk, score=round(s, 4), note=note)

In [ ]:
import shutil
arrs = {}
for kind, (tfs, clf) in TEXT_MODELS.items():
    for n, t in tfs.items(): arrs[f'{kind}__{n}__idf'] = t.idf_.astype(np.float32)
    arrs[f'{kind}__coef'] = clf.coef_.ravel().astype(np.float32); arrs[f'{kind}__intercept'] = clf.intercept_.astype(np.float64)
np.savez_compressed(OUT / 'text_models.npz', **arrs)
GBM.save_model(str(OUT / 'gbm.txt'), num_iteration=BEST)
json.dump({'lm_tri': LM.tri, 'lm_bi': LM.bi, 'tld_index': TLD_INDEX}, open(OUT / 'resources.json', 'w'))
for f in ('mlphil_urlcore.py', 'mlphil_features.py', 'mlphil_predictor.py'): shutil.copy(f, OUT / f)
json.dump(dict(name='mlphil_url_risk', created_utc=datetime.now(timezone.utc).isoformat(timespec='seconds'),
               feats=FEATS, thr_high=THR_HIGH, thr_medium=THR_MED, best_iteration=BEST,
               text_blocks={k: list(v) for k, v in TEXT.items()}, config=CFG, verdict=VERDICT,
               data_provenance=PROV), open(OUT / 'metadata.json', 'w'), indent=1, default=str)

# parity: exported bundle must reproduce in-notebook scores
sys.path.insert(0, str(OUT)); import mlphil_predictor; importlib.reload(mlphil_predictor)
model = mlphil_predictor.URLRisk(OUT)
chk = ext.sample(min(2000, len(ext)), random_state=4)
diff = np.nanmax(np.abs(model.score_many(chk.raw.tolist()) - chk.score.values))
print(f'parity max |Δ| = {diff:.2e} ->', 'PASS' if diff < 1e-6 else 'FAIL')
# ---- Website compatibility -------------------------------------------------------------------
# The ML Phil site's url_v5_service.py imports `predictor.URLRiskV5`, `urlcore` and `features`
# and reads thr_alert / thr_review. These thin wrappers expose this bundle under those names;
# the model code itself is unchanged (mlphil_*.py).
(OUT / 'urlcore.py').write_text('"""Website name for mlphil_urlcore (same code)."""\nfrom mlphil_urlcore import *  # noqa: F401,F403\n')
(OUT / 'features.py').write_text('"""Website name for mlphil_features (same code)."""\nfrom mlphil_features import *  # noqa: F401,F403\n')
(OUT / 'predictor.py').write_text('''"""Website entry point: same interface as the v5 bundle (URLRiskV5, thr_alert, thr_review)."""
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from mlphil_predictor import URLRisk


class URLRiskV5(URLRisk):
    def __init__(self, bundle_dir):
        super().__init__(bundle_dir)
        self.thr_alert, self.thr_review = self.thr_high, self.thr_med
''')
meta = json.load(open(OUT / 'metadata.json'))
meta.update(version='mlphil_url_v6', thr_alert=THR_HIGH, thr_review=THR_MED)
json.dump(meta, open(OUT / 'metadata.json', 'w'), indent=1, default=str)

# Prove it in a fresh interpreter: the website's imports work, and every feature name its evidence list reads exists.
import subprocess
site_check = f'''
import sys; sys.path.insert(0, {str(OUT)!r})
from predictor import URLRiskV5
import urlcore as U
m = URLRiskV5({str(OUT)!r})
need = set("is_ip userinfo at_in_path punycode brand_in_sub brand_in_path brand_in_sld_mixed lure_words_host lure_words_path "
           "platform shortener embedded_url sub_labels has_port file_ext path_depth lookalike_d1 lookalike_d2 brand_token_in_host lm_sld".split())
missing = need - set(U.NUM_FEATURES) - set(U.EXTRA_FEATURES)
assert not missing, f"evidence features missing: {{missing}}"
for u in ["paypal.com.secure-verify.xyz/signin", "https://kiit.ac.in/"]:
    r = m.assess(u); assert r["risk"] in {{"HIGH", "MEDIUM", "LOW"}}, r
    p = U.parse(u)
    f = dict(zip(U.NUM_FEATURES, U.numeric(p))); x = dict(zip(U.EXTRA_FEATURES, U.extra(p, m.lm, m.tld_index)))
    assert f["is_ip"] is not None and x["lm_sld"] is not None
assert m.assess("javascript:alert(1)")["risk"] == "INVALID"
print("website compatibility OK | thr_alert", round(m.thr_alert, 4), "| thr_review", round(m.thr_review, 4))
'''
res = subprocess.run([sys.executable, "-c", site_check], capture_output=True, text=True, timeout=600)
print(res.stdout.strip() or res.stderr[-2000:]); assert res.returncode == 0, "bundle does not fit the website"

# External test URLs with labels and scores, so the website's current model can be scored on exactly the same URLs.
ext.loc[ext.split == 'ext_test', ['raw', 'source', 'label', 'score']].to_csv(WORK / 'ext_test_urls.csv.gz', index=False)
print('saved ext_test_urls.csv.gz:', int((ext.split == 'ext_test').sum()), 'URLs')

shutil.make_archive(str(WORK / 'model_bundle'), 'zip', OUT)
print('bundle:', sorted(os.listdir(OUT)), '| zip:', WORK / 'model_bundle.zip')

In [ ]:
for u in ['https://www.google.com/', 'https://kiit.ac.in/', 'https://www.irctc.co.in/nget/train-search',
          'http://sbi-kyc-update.online/login.php', 'paypal.com.secure-verify.xyz/signin',
          'https://my-office365-docs.pages.dev/auth', 'https://appleid.apple.com.verify-id.co/']:
    print(f'{u:50s}', model.assess(u))

## 12. Summary

In [ ]:
summary = RESULTS[RESULTS.split.isin(['test', 'ext_test'])][['split', 'cohort', 'metric', 'n', 'rate_HIGH', 'ci_lo', 'ci_hi']]
display(summary.round(4))
print(f'Thresholds: HIGH >= {THR_HIGH:.4f}, MEDIUM >= {THR_MED:.4f}')
print('Canonical invariance max |Δ|:', float(INV.max()))
print('Seed stability (max std across test cohorts):', round(float(SEED_SUM.seed_std.max()), 4))
print('Grouped CV AUC: %.4f ± %.4f' % (CV.AUC.mean(), CV.AUC.std()))
print('VERDICT:', VERDICT)
print(f'total runtime {(time.time()-T0)/60:.1f} min')
json.dump(dict(verdict=VERDICT, thr_high=THR_HIGH, thr_medium=THR_MED, criteria=CHECK.to_dict('index'),
               results=RESULTS.to_dict('records'), invariance=INV.to_dict(), perturbation=PERT.to_dict('records'),
               seed_stability=SEED_SUM.reset_index().to_dict('records'), grouped_cv=CV.to_dict('records'),
               bootstrap=BOOT.to_dict('records')), open(WORK / 'final_report.json', 'w'), indent=1, default=float)

# Ship the report inside the bundle (the website reads its figures from there) and rebuild the zip.
shutil.copy(WORK / 'final_report.json', OUT / 'final_report.json')
shutil.make_archive(str(WORK / 'model_bundle'), 'zip', OUT)
print('final_report.json added to model_bundle/ and model_bundle.zip rebuilt')

**Limitations (state these in the report):** benign cohorts are proxies (popularity lists, curated links), phishing cohorts are feed-listed;
Phishing.Database test shares a feed family with part of the training data (domain-disjoint only); bare phishing domains are harder (URL text carries little signal);
real brand pages with account-style paths can trip HIGH. Use as an **advisory** signal next to threat-intelligence lookups — never as a "safe" verdict.